# 01B. Construcción del subcorpus normativo y operativo

Este notebook trabaja sobre las fuentes jurídicas, políticas y de manejo previamente registradas en el corpus general. No mantiene un registro documental independiente.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import yaml

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'config' / 'taxonomy.yml').exists():
            return candidate
    raise FileNotFoundError('No se encontró config/taxonomy.yml')

ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from regulatory_review.schema import RegulatoryRecord
from regulatory_review.scoring import score_transferability

print(f'Project root: {ROOT}')


## 1. Cargar taxonomía y registro maestro

In [ ]:
with (ROOT / 'config' / 'taxonomy.yml').open(encoding='utf-8') as stream:
    taxonomy = yaml.safe_load(stream)

registry_path = ROOT / 'data' / 'templates' / 'source_registry.csv'
registry = pd.read_csv(registry_path)

print(taxonomy['project']['name'])
print('Management measures:', len(taxonomy['management_measure']))
print('Trigger types:', len(taxonomy['trigger_type']))
print('Registered sources:', len(registry))


## 2. Seleccionar fuentes normativas y operativas

El subcorpus incluye instrumentos jurídicos y documentos de gestión que pueden contener mandatos, procedimientos, disparadores o medidas. Los artículos científicos permanecen en el corpus general, incluso cuando analicen estas normas.

In [ ]:
regulatory_types = {
    'international_agreement', 'regional_measure', 'national_law',
    'regulation', 'policy', 'strategy', 'adaptation_plan',
    'fishery_management_plan', 'harvest_strategy',
    'technical_protocol', 'guideline_or_manual'
}

if registry.empty:
    regulatory_sources = registry.copy()
else:
    regulatory_sources = registry.loc[registry['source_type'].isin(regulatory_types)].copy()

regulatory_sources.head()


## 3. Auditar metadatos requeridos para interpretación jurídica

In [ ]:
required_columns = {
    'source_id', 'title', 'source_type', 'year', 'jurisdiction',
    'source_status', 'legal_status', 'primary_url', 'version_date',
    'access_date', 'source_language', 'local_path'
}
missing_columns = sorted(required_columns - set(registry.columns))
if missing_columns:
    raise ValueError(f'Faltan columnas: {missing_columns}')

if not regulatory_sources.empty:
    duplicated = regulatory_sources.loc[
        regulatory_sources['source_id'].duplicated(keep=False), 'source_id'
    ].tolist()
    if duplicated:
        raise ValueError(f'source_id duplicados: {duplicated}')

    missing_legal_status = regulatory_sources['legal_status'].isna().sum()
    print('Fuentes sin legal_status:', missing_legal_status)

print(f'Subcorpus normativo: {len(regulatory_sources)} fuentes')


## 4. Ejemplo de fuente candidata

Esta fila sigue el registro maestro. No debe añadirse hasta verificar la versión primaria, la jurisdicción y la URL.

In [ ]:
candidate_source = {
    'source_id': 'jurisdiction_year_short_title',
    'title': '',
    'source_type': 'fishery_management_plan',
    'source_subtype': '',
    'authors_or_organisation': '',
    'publisher': '',
    'year': None,
    'version_date': '',
    'jurisdiction': '',
    'geographic_scope': '',
    'fishery_scope': '',
    'species': '',
    'source_status': 'official_non_binding',
    'legal_status': 'unclear',
    'doi': '',
    'primary_url': '',
    'landing_page_url': '',
    'access_date': '',
    'source_language': '',
    'local_path': '',
    'checksum_sha256': '',
    'supersedes_source_id': '',
    'notes': '',
}
pd.DataFrame([candidate_source])


## 5. Transferibilidad regulatoria

Esta puntuación prioriza mecanismos para revisión. No demuestra efectividad ni reemplaza el análisis jurídico comparado.

In [ ]:
example_score = score_transferability(
    ecological_similarity=3,
    climate_operationalisation=2,
    trigger_specificity=2,
    response_predefinition=2,
    legal_force=2,
    data_feasibility_peru=2,
)
example_score


## Próximos pasos

1. Registrar primero la fuente en el corpus general.
2. Extraer artículos, secciones, anexos o reglas.
3. Ejecutar el cribado normativo especializado.
4. Extraer mecanismos con `RegulatoryRecord`.
5. Validar vigencia, fuerza jurídica, localizador y extracto.